<a href="https://colab.research.google.com/github/ajayrfhp/LearningDeepLearning/blob/main/MixtureOfExperts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Core problem
- Historically, more parameters = more learning capacity for model = more compute.
- How to get performance without compute ? https://arxiv.org/abs/1701.06538

## Answer
- Conditional computation. Activate parts of model depending on input

## Step 1: Warm-up Code Challenge

Imagine you are designing a simple routing layer
in Python/PyTorch.

Suppose you have $N = 4$ sub-networks (experts), each represented by a simple matrix multiplication. You want to compute the output for a single input vector $x$.

1. Dense Routing: Write a line of code computing gate weights g using a standard softmax(x @ W_g). How many experts get evaluated?

In [22]:
import torch
import torch.nn as nn


experts = torch.stack([torch.randn((20, 10)) for _ in range(4)]) # (N, DH, DM)
w_g = torch.randn((10, 4)) # (DM, N)
x = torch.randn((1, 10)) # (M, DM)

N = experts.shape[0]
M = x.shape[0]
DM = x.shape[1]
DH = experts.shape[1]

In [26]:
# dense_routing (M, DM) @ (DM, N) = (M, N) => (M, N, 1)
g = nn.Softmax(dim=-1)(x @ w_g).unsqueeze(-1) # (1*10*4 computations) # shape of g is (4)

assert g.shape == (M, N, 1)

# (N, DH, M) = (N, DH, 1, DM) @ (DM, M)
a = (experts @ x.permute(1, 0))

assert a.shape == (N, DH, M)

# (M, N, 1) @ (M, N, DH) = (M, N, DH) => (M, DH)
weighted_experts_matmul = (g * a.permute(2, 0, 1)).sum(1)

assert weighted_experts_matmul.shape == (M, DH)


# (M, DM) @ (DM, N) = (M, N)
g = torch.softmax(x @ w_g, dim=-1)

# (ndm)         (om) => (nod)
# (N, DH, DM) @ (M, DM)= (N, M, DH)
a = torch.einsum("ndm,om->nod", experts, x)

assert a.shape == (N, M, DH)

# (M, N, 1) @ (M, N, DH) = (M, N, DH)
weighted_experts_einsum = torch.einsum("mn,nmh->mh",g,a)

assert weighted_experts_einsum.shape == (M, DH)

assert torch.allclose(weighted_experts_matmul, weighted_experts_einsum)
